# Week 8 Lab 1: LeNet-5 Architecture Implementation (Keras)

> **Goal**: Implement the foundational LeNet-5 Convolutional Neural Network (CNN) to recognize handwritten digits using TensorFlow and Keras, and export the model for Edge deployment.

## 1. Setup and Load Data

**Why it Matters**: A CNN requires the data to have spatial dimensionality. For MNIST, we must reshape our 1D arrays into 3D tensors: `[Height=28, Width=28, Channels=1]`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, datasets
import matplotlib.pyplot as plt

# Load MNIST dataset
(train_images, train_labels), (test_images, test_labels) = datasets.mnist.load_data()

# Reshape to include the Channel dimension (Channel-Last format for TF)
train_images = train_images.reshape((60000, 28, 28, 1)) / 255.0
test_images = test_images.reshape((10000, 28, 28, 1)) / 255.0

print(f"Train Data Shape: {train_images.shape}")

## 2. Building LeNet-5

**Concept**: The LeNet-5 architecture alternates between **Convolutional Layers** (to extract features) and **Pooling Layers** (to compress spatial resolution). Finally, it connects to **Dense Layers** for classification.

**Task**: Run the cell below to define the architecture.

In [ ]:
model = models.Sequential([
    # Block 1: Feature Extraction
    layers.Conv2D(6, kernel_size=(5, 5), activation='relu', input_shape=(28, 28, 1), padding='same'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Block 2: Feature Extraction
    layers.Conv2D(16, kernel_size=(5, 5), activation='relu'),
    layers.MaxPooling2D(pool_size=(2, 2)),
    
    # Flatten and Classify
    layers.Flatten(),
    layers.Dense(120, activation='relu'),
    layers.Dense(84, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()

**Observation**: Look at the Output Shapes in the summary. Notice how the height and width (the middle two numbers) get smaller due to Pooling, but the number of filters (the last number) increases due to Convolution.

## 3. Training the Model

**Task**: Compile and fit the model to the training data. This takes significantly less code than raw framework implementations.

In [ ]:
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

history = model.fit(train_images, train_labels, epochs=5, 
                    validation_data=(test_images, test_labels))

## 4. Evaluate and Visualize Predictions

Let's see the CNN in action!

In [ ]:
import numpy as np

# Predict the first 5 test images
predictions = model.predict(test_images[:5])

plt.figure(figsize=(10, 5))
for i in range(5):
    plt.subplot(1, 5, i+1)
    plt.imshow(test_images[i].reshape(28, 28), cmap='gray')
    plt.title(f"Pred: {np.argmax(predictions[i])}")
    plt.axis('off')
plt.show()

## 5. Export for Jetson Edge Deployment (TensorFlow Lite)

To deploy this CNN on an embedded edge device like the **Jetson Orin Nano**, we should convert it into a highly optimized format called **TensorFlow Lite (.tflite)**. 

**Task**: Run the cell below to export your trained model.

In [ ]:
# Convert the model
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model to disk
with open('lenet_mnist.tflite', 'wb') as f:
    f.write(tflite_model)

print("Model successfully exported to: lenet_mnist.tflite")
print("You can now transfer this file to your Jetson device!")